In [1]:
# install libraries
!pip install langchain langchain-groq

In [2]:
# importing libraries
from langchain_groq import ChatGroq
from langchain.tools import tool

In [3]:
# chat model 
import os

os.environ["GROQ_API_KEY"] = "gsk_
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [4]:
# testing chat model 
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [5]:
# custom tool 1 - addition 
@tool
def add(a: int, b: int) -> int:
    """
    Adds two integers together.

    Args:
        a: First integer
        b: Second integer

    Returns:
        Sum of a and b
    """
    return a + b

In [6]:
# testing tool 
add.invoke({"a": 5, "b": 7})

12

In [7]:
# binding custom tool 1 to llm 
llm_with_tools = llm.bind_tools([add])

In [8]:
# testing tool calling
response = llm_with_tools.invoke(
    "What is 25 plus 17?"
)

response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8bnkzcnnz', 'function': {'arguments': '{"a":25,"b":17}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 273, 'total_tokens': 291, 'completion_time': 0.049562425, 'completion_tokens_details': None, 'prompt_time': 0.01464701, 'prompt_tokens_details': None, 'queue_time': 0.187786709, 'total_time': 0.064209435}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019ef49f-a470-7a10-b369-fea3a44a488c-0', tool_calls=[{'name': 'add', 'args': {'a': 25, 'b': 17}, 'id': '8bnkzcnnz', 'type': 'tool_call'}], usage_metadata={'input_tokens': 273, 'output_tokens': 18, 'total_tokens': 291})

In [9]:
# parss and run tool call 
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

print(tool_name)
print(tool_args)

add
{'a': 25, 'b': 17}


In [10]:
result = add.invoke(tool_args)
print(result)

42


In [11]:
# tool mapping dictionary 
tools = [add]

tool_map = {tool.name: tool for tool in tools}

tool_map

{'add': StructuredTool(name='add', description='Adds two integers together.\n\nArgs:\n    a: First integer\n    b: Second integer\n\nReturns:\n    Sum of a and b', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x7fa33a2680d0>)}

In [12]:
# testing mapping
result = tool_map[tool_name].invoke(tool_args)
print(result)

42


In [13]:
# custom tool 2 - multiply
@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a: First integer
        b: Second integer

    Returns:
        Product of a and b
    """
    return a * b

In [14]:
# testing tool
multiply.invoke({"a": 6, "b": 7})

42

In [15]:
# custom tool 3 - subtract
@tool
def subtract(a: int, b: int) -> int:
    """
    Subtracts the second integer from the first integer.

    Args:
        a: First integer
        b: Second integer

    Returns:
        Difference between a and b
    """
    return a - b

In [16]:
# testing tool
subtract.invoke({"a": 10, "b": 4})

6

In [17]:
# custom tool 4 - divide
@tool
def divide(a: float, b: float) -> float:
    """
    Divides the first number by the second number.

    Args:
        a: Numerator
        b: Denominator

    Returns:
        Quotient of a and b
    """
    if b == 0:
        return "Cannot divide by zero."

    return a / b

In [18]:
# testing tool 
divide.invoke({"a": 20, "b": 5})

4.0

In [32]:
# tools in a list
tools = [add, subtract, multiply, divide, calculate_tip]

In [33]:
# mapping dictionary
tool_map = {tool.name: tool for tool in tools}

In [34]:
# checking dictionary 
tool_map.keys()

dict_keys(['add', 'subtract', 'multiply', 'divide', 'calculate_tip'])

In [35]:
# bind all tools to llm 
llm_with_tools = llm.bind_tools(tools)

In [23]:
# testing tool selection 
response = llm_with_tools.invoke(
    "What is 8 multiplied by 7?"
)

response.tool_calls

[{'name': 'multiply',
  'args': {'a': 8, 'b': 7},
  'id': 'zmdff6n72',
  'type': 'tool_call'}]

In [26]:
# checking tools output
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

result = tool_map[tool_name].invoke(tool_args)

print(result)

56


In [25]:
from langchain_core.messages import HumanMessage, ToolMessage

In [27]:
# generate final answer from chat history 
messages = [
    HumanMessage(content="What is 8 multiplied by 7?")
]

response = llm_with_tools.invoke(messages)

tool_call = response.tool_calls[0]
tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])

messages.append(response)

messages.append(
    ToolMessage(
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )
)

final_response = llm_with_tools.invoke(messages)

print(final_response.content)

# user question -> LLM picked tool -> tool calculated answer -> LLM gives final answer

The answer is 56.


In [28]:
# reusable agent function 
def run_tool_agent(user_query: str):
    messages = [HumanMessage(content=user_query)]

    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        return response.content

    messages.append(response)

    for tool_call in response.tool_calls:
        selected_tool = tool_map[tool_call["name"]]
        tool_result = selected_tool.invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call["id"]
            )
        )

    final_response = llm_with_tools.invoke(messages)
    return final_response.content

In [29]:
# testing 
run_tool_agent("What is 100 divided by 4?")

'The result of 100 divided by 4 is 25.'

In [30]:
# tip calculator tool 
@tool
def calculate_tip(bill_amount: float, tip_percent: float) -> float:
    """
    Calculates the tip amount for a bill.

    Args:
        bill_amount: Total bill amount
        tip_percent: Tip percentage

    Returns:
        Tip amount
    """
    return bill_amount * (tip_percent / 100)

In [31]:
calculate_tip.invoke({
    "bill_amount": 50,
    "tip_percent": 15
})

7.5

In [36]:
# testing to see if llm uses new tool 
response = llm_with_tools.invoke(
    "Calculate a 20 percent tip on a $75 bill."
)

response.tool_calls

[{'name': 'calculate_tip',
  'args': {'bill_amount': 75, 'tip_percent': 20},
  'id': 'dh93tyq9j',
  'type': 'tool_call'}]

In [37]:
# testing reusable agent
run_tool_agent("Calculate a 20 percent tip on a $75 bill.")

'The tip amount for a $75 bill with a 20% tip is $15.00.'